### 任务见codespace

In [1]:
import os
import torch

# 配置 HuggingFace 镜像，防止国内下载模型超时
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# 检测 GPU 是否可用
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"当前使用设备: {device}")
if device == "cuda":
    print(f"显卡型号: {torch.cuda.get_device_name(0)}")

当前使用设备: cuda
显卡型号: NVIDIA GeForce RTX 5060 Laptop GPU


In [2]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings # 若报错可改回 from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# 1. 示例文本转换为 Document 对象
texts = [
    "张三是法外狂徒",
    "FAISS是一个用于高效相似性搜索和密集向量聚类的库。",
    "LangChain是一个用于开发由语言模型驱动的应用程序的框架。"
]
docs = [Document(page_content=t) for t in texts]

# 2. 在 GPU 上加载嵌入模型
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-zh-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)
print("嵌入模型加载完成！")

C:\Users\35368\AppData\Local\Temp\ipykernel_15984\3265368449.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\miniconda\envs\pytorch\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\35368\.cache\huggingface\hub\models--BAAI--bge-small-zh-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/27.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

... (7 outputs left out)

tokenizer.json:   0%|          | 0.00/439k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

嵌入模型加载完成！


In [3]:
# 构建向量存储并持久化到本地
vectorstore = FAISS.from_documents(docs, embeddings)

local_faiss_path = "./faiss_index_store"
vectorstore.save_local(local_faiss_path)

print(f"FAISS 索引已保存至: {local_faiss_path}")
print("生成的文件:", os.listdir(local_faiss_path))

FAISS 索引已保存至: ./faiss_index_store
生成的文件: ['index.faiss', 'index.pkl']


In [4]:
# 加载本地索引（需传入相同的 embeddings，并允许反序列化 .pkl 文件）
loaded_vectorstore = FAISS.load_local(
    local_faiss_path,
    embeddings,
    allow_dangerous_deserialization=True
)

# 相似性搜索
query = "FAISS是做什么的？"
results = loaded_vectorstore.similarity_search(query, k=1)

print(f"查询: '{query}'")
print("相似度最高的文档:")
for doc in results:
    print(f"- {doc.page_content}")

查询: 'FAISS是做什么的？'
相似度最高的文档:
- FAISS是一个用于高效相似性搜索和密集向量聚类的库。


In [5]:
import os
import torch
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 确保使用国内镜像与 GPU
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. 配置全局嵌入模型（必须设置，否则默认尝试调用 OpenAI）
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-zh-v1.5",
    device=device
)

# 2. 创建示例文档
texts = [
    "张三是法外狂徒",
    "FAISS是一个用于高效相似性搜索和密集向量聚类的库。",
    "LlamaIndex是一个用于构建和查询私有或领域特定数据的框架。",
    "它提供了数据连接、索引和查询接口等工具。"
]
docs = [Document(text=t) for t in texts]

# 3. 创建索引并持久化到本地
index = VectorStoreIndex.from_documents(docs)
persist_path = "./llamaindex_index_store"
index.storage_context.persist(persist_dir=persist_path)

print(f"LlamaIndex 索引已保存至: {persist_path}")
print("生成的 JSON 文件:", os.listdir(persist_path))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\miniconda\envs\pytorch\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\35368\AppData\Local\llama_index\llama_index\Cache\models--BAAI--bge-small-zh-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/27.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/439k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

LlamaIndex 索引已保存至: ./llamaindex_index_store
生成的 JSON 文件: ['default__vector_store.json', 'docstore.json', 'graph_store.json', 'image__vector_store.json', 'index_store.json']


In [6]:
import json

# 查看 docstore.json（存储原始文本和节点元数据）
with open(os.path.join(persist_path, "docstore.json"), "r", encoding="utf-8") as f:
    docstore_data = json.load(f)
print("=== docstore.json 预览 ===")
print(json.dumps(docstore_data, ensure_ascii=False, indent=2)[:800], "\n...")

# 查看 default__vector_store.json（存储向量数组与节点 ID 映射）
with open(os.path.join(persist_path, "default__vector_store.json"), "r", encoding="utf-8") as f:
    vector_data = json.load(f)
print("\n=== default__vector_store.json 包含的键 ===")
print(list(vector_data.keys()))
print("存储的向量数量:", len(vector_data["embedding_dict"]))

=== docstore.json 预览 ===
{
  "docstore/metadata": {
    "9486d516-0d38-4e08-a181-f3f04f1b4e3d": {
      "doc_hash": "5a5ecf71785a042d2b31988694b59c7925094eaea273e467a6afe776a42432cd"
    },
    "7bb64363-e4a7-40a3-be93-5b7c185f89e9": {
      "doc_hash": "3555afa6f407e915159b71b00878597750c1ce8e6c8027aa68a91c2a0eef7ff9"
    },
    "29df5b49-cfd8-4927-bc7f-4b5d1d091998": {
      "doc_hash": "4ab57ddb6b0b7726615f0e027766bff97651c33aaa4ca8e48755f6c8476c6ca0"
    },
    "614486e1-592a-4175-93ca-1b5cd2573e13": {
      "doc_hash": "d5ab17a353a3daf441a93a172c9351bc014c6c87763ff7b8eca8a01a9cd1d035"
    },
    "51f7a83c-4cc4-4fbe-ab5e-2d62aa2fc36a": {
      "doc_hash": "bf8d2f004fc7155822678e4e01cdcbc848dc8999d91493b69d94dd23127d96f3",
      "ref_doc_id": "9486d516-0d38-4e08-a181-f3f04f1b4e3d"
    },
    "c95d0b09-04ca-4527 
...

=== default__vector_store.json 包含的键 ===
['embedding_dict', 'text_id_to_ref_doc_id', 'metadata_dict']
存储的向量数量: 4


In [7]:
import os
import torch
from llama_index.core import StorageContext, load_index_from_storage, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 1. 配置环境与相同的嵌入模型（加载和查询时必须与构建索引的模型一致）
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
device = "cuda" if torch.cuda.is_available() else "cpu"

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-zh-v1.5",
    device=device
)

# 2. 从本地持久化目录加载索引
persist_path = "./llamaindex_index_store"
storage_context = StorageContext.from_defaults(persist_dir=persist_path)
loaded_index = load_index_from_storage(storage_context)
print("索引加载成功！")

# 3. 创建检索器并执行相似性搜索（仅做向量检索，不调用 LLM 生成回答）
retriever = loaded_index.as_retriever(similarity_top_k=2)

query = "LlamaIndex是用来做什么的？"
results = retriever.retrieve(query)

print(f"\n查询: '{query}'")
print("相似度最高的文档:")
for i, node_with_score in enumerate(results, 1):
    print(f"{i}. [相似度得分: {node_with_score.score:.4f}] {node_with_score.node.text}")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

索引加载成功！

查询: 'LlamaIndex是用来做什么的？'
相似度最高的文档:
1. [相似度得分: 0.6761] LlamaIndex是一个用于构建和查询私有或领域特定数据的框架。
2. [相似度得分: 0.5287] FAISS是一个用于高效相似性搜索和密集向量聚类的库。
